In [1]:
import json

try:
    levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
except FileNotFoundError:
    levels = [
        {"concurrency": 1,  "tokens_per_s": 38.2,  "latency_p95_s": 0.9,  "errors": 0},
        {"concurrency": 2,  "tokens_per_s": 71.5,  "latency_p95_s": 1.1,  "errors": 0},
        {"concurrency": 4,  "tokens_per_s": 128.4, "latency_p95_s": 1.4,  "errors": 0},
        {"concurrency": 8,  "tokens_per_s": 210.7, "latency_p95_s": 2.3,  "errors": 0},
        {"concurrency": 16, "tokens_per_s": 224.9, "latency_p95_s": 5.8,  "errors": 0},
    ]
    print("using the sample bench_report -- swap in your own file for a real answer")

for L in levels:
    print(L)

{'concurrency': 1, 'tokens_per_s': 72.75, 'ttft_p50_s': 0.0846, 'ttft_p95_s': 0.3234, 'latency_p95_s': 2.3857, 'errors': 0, 'ok': 20, 'wall_s': 27.533}
{'concurrency': 2, 'tokens_per_s': 169.74, 'ttft_p50_s': 0.0605, 'ttft_p95_s': 0.0918, 'latency_p95_s': 1.7021, 'errors': 0, 'ok': 20, 'wall_s': 11.8}
{'concurrency': 4, 'tokens_per_s': 281.15, 'ttft_p50_s': 0.0593, 'ttft_p95_s': 0.1155, 'latency_p95_s': 1.7949, 'errors': 0, 'ok': 20, 'wall_s': 7.124}
{'concurrency': 8, 'tokens_per_s': 431.7, 'ttft_p50_s': 0.1417, 'ttft_p95_s': 0.4328, 'latency_p95_s': 2.3658, 'errors': 0, 'ok': 20, 'wall_s': 4.64}
{'concurrency': 16, 'tokens_per_s': 701.61, 'ttft_p50_s': 0.2428, 'ttft_p95_s': 0.2473, 'latency_p95_s': 2.3631, 'errors': 0, 'ok': 20, 'wall_s': 2.959}


In [2]:
def cost_per_million_tokens(tokens_per_s, gpu_hourly_usd):
    tokens_per_hour = tokens_per_s * 3600
    million_tokens_per_hour = tokens_per_hour / 1_000_000
    return round(gpu_hourly_usd / million_tokens_per_hour, 4)

GPU_HOURLY_USD = 0.35   # a representative on-demand T4-class price; swap in your real rate

for L in levels:
    L["cost_per_million_tokens_usd"] = cost_per_million_tokens(L["tokens_per_s"], GPU_HOURLY_USD)

for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"p95={L['latency_p95_s']:.2f}s  $/M tok=${L['cost_per_million_tokens_usd']}")

c= 1  tok/s=   72.8  p95=2.39s  $/M tok=$1.3364
c= 2  tok/s=  169.7  p95=1.70s  $/M tok=$0.5728
c= 4  tok/s=  281.1  p95=1.79s  $/M tok=$0.3458
c= 8  tok/s=  431.7  p95=2.37s  $/M tok=$0.2252
c=16  tok/s=  701.6  p95=2.36s  $/M tok=$0.1386


In [3]:
TARGET_P95_S = 5.0   #  Predictive SLOs

under_target = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under_target, key=lambda L: L["concurrency"]) if under_target else None
print("knee:", knee)

past_knee = [L for L in levels if knee and L["concurrency"] > knee["concurrency"]]
if past_knee:
    cheapest_past_knee = min(past_knee, key=lambda L: L["cost_per_million_tokens_usd"])
    print("cheapest $/M token level past the knee (SLO-violating):", cheapest_past_knee)
    print("-> cheaper on paper, but its p95 already exceeds your SLO -- "
          "not real usable capacity at your target.")

knee: {'concurrency': 16, 'tokens_per_s': 701.61, 'ttft_p50_s': 0.2428, 'ttft_p95_s': 0.2473, 'latency_p95_s': 2.3631, 'errors': 0, 'ok': 20, 'wall_s': 2.959, 'cost_per_million_tokens_usd': 0.1386}


In [4]:
import math

def replicas_needed(required_tokens_per_s, knee_tokens_per_s):
    return math.ceil(required_tokens_per_s / knee_tokens_per_s)

def scale_out_cost(required_tokens_per_s, knee, gpu_hourly_usd):
    n = replicas_needed(required_tokens_per_s, knee["tokens_per_s"])
    return {
        "required_tokens_per_s": required_tokens_per_s,
        "replicas_needed": n,
        "total_hourly_cost_usd": round(n * gpu_hourly_usd, 2),
        "effective_p95_s": knee["latency_p95_s"],   # every replica runs at the same safe knee
    }

targets = [knee["tokens_per_s"] * m for m in (1.0, 1.5, 2.0, 3.0)]
scale_plan = [scale_out_cost(t, knee, GPU_HOURLY_USD) for t in targets]
for row in scale_plan:
    print(row)

{'required_tokens_per_s': 701.61, 'replicas_needed': 1, 'total_hourly_cost_usd': 0.35, 'effective_p95_s': 2.3631}
{'required_tokens_per_s': 1052.415, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 2.3631}
{'required_tokens_per_s': 1403.22, 'replicas_needed': 2, 'total_hourly_cost_usd': 0.7, 'effective_p95_s': 2.3631}
{'required_tokens_per_s': 2104.83, 'replicas_needed': 3, 'total_hourly_cost_usd': 1.05, 'effective_p95_s': 2.3631}


In [5]:
report = {
    "gpu_hourly_usd": GPU_HOURLY_USD,
    "target_p95_s": TARGET_P95_S,
    "levels": levels,
    "knee": knee,
    "scale_out_plan": scale_plan,
}
with open("cost_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

{
  "gpu_hourly_usd": 0.35,
  "target_p95_s": 5.0,
  "levels": [
    {
      "concurrency": 1,
      "tokens_per_s": 72.75,
      "ttft_p50_s": 0.0846,
      "ttft_p95_s": 0.3234,
      "latency_p95_s": 2.3857,
      "errors": 0,
      "ok": 20,
      "wall_s": 27.533,
      "cost_per_million_tokens_usd": 1.3364
    },
    {
      "concurrency": 2,
      "tokens_per_s": 169.74,
      "ttft_p50_s": 0.0605,
      "ttft_p95_s": 0.0918,
      "latency_p95_s": 1.7021,
      "errors": 0,
      "ok": 20,
      "wall_s": 11.8,
      "cost_per_million_tokens_usd": 0.5728
    },
    {
      "concurrency": 4,
      "tokens_per_s": 281.15,
      "ttft_p50_s": 0.0593,
      "ttft_p95_s": 0.1155,
      "latency_p95_s": 1.7949,
      "errors": 0,
      "ok": 20,
      "wall_s": 7.124,
      "cost_per_million_tokens_usd": 0.3458
    },
    {
      "concurrency": 8,
      "tokens_per_s": 431.7,
      "ttft_p50_s": 0.1417,
      "ttft_p95_s": 0.4328,
      "latency_p95_s": 2.3658,
      "errors": 0,
   

In [6]:
!python verify.py

recomputed costs, knee and scale-out plan all agree
GREEN CHECK: PASS
